<a href="https://colab.research.google.com/github/piyushsinhrajput/ML-7218-6D/blob/main/P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_csv("/content/01 covid_toy.csv")

display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [ ]:
df.select_dtypes(include= np.number).columns

Index(['age', 'fever'], dtype='object')

In [ ]:
numeric_columns = df.select_dtypes(include= np.number).columns

print("Number Columns: ")
print(numeric_columns)


Number Columns: 
Index(['age', 'fever'], dtype='object')


In [ ]:
from pandas.core.arrays import categorical
categorical_columns = df.select_dtypes(include="object").columns

print("Categorical Columns: ")
print(categorical_columns)


Categorical Columns: 
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [ ]:
print(df.dtypes)


age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [ ]:
df.select_dtypes(include= np.number).columns

Index(['age', 'fever'], dtype='object')

In [ ]:
numeric_columns = df.select_dtypes(include= np.number).columns

print("Number Columns: ")
print(numeric_columns)

Number Columns: 
Index(['age', 'fever'], dtype='object')


In [ ]:
from pandas.core.arrays import categorical
categorical_columns = df.select_dtypes(include="object").columns

print("Categorical Columns: ")
print(categorical_columns)

Categorical Columns: 
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [ ]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [ ]:
df = df.replace("?",np.nan)

In [ ]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [ ]:
# Clean up gender by stripping whitespace and matching the casing
df["gender"] = df["gender"].str.strip()
df["gender"] = df["gender"].map({"Female": 0, "Male": 1})

In [ ]:
# Fill any remaining missing values in gender (though there shouldn't be any now)
df["gender"] = df["gender"].fillna(df["gender"].mode()[0])

# Also impute the 'fever' column so we have no NaNs in numeric columns
df["fever"] = df["fever"].fillna(df["fever"].median())

In [ ]:
print(df.isnull().sum())

age            0
gender       100
fever         10
cough          0
city           0
has_covid      0
dtype: int64


In [ ]:
scaler = MinMaxScaler().set_output(transform="pandas")

scaled_data = scaler.fit_transform(df[numeric_columns])

scaled_data.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [ ]:
categorical_columns = ["city","has_covid"]

In [ ]:
encode = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
).set_output(transform="pandas")

encoded_data = encode.fit_transform(df[categorical_columns])

display(encoded_data.head())

,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai,has_covid_No,has_covid_Yes
0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,1.0,0.0


In [ ]:
x = df.drop("age",axis=1)
y = df["age"]

In [ ]:
raw_df = pd.read_csv("/content/01 covid_toy.csv")
df["gender"] = raw_df["gender"].map({"Female": 0, "Male": 1})

x = df.drop("age", axis=1)
y = df["age"]

X_numeric = x.select_dtypes(include=np.number)
imputer = SimpleImputer(strategy="mean").set_output(transform="pandas")
X_numeric_imputed = imputer.fit_transform(X_numeric)

selector = SelectKBest(
    score_func=f_regression,
    k=2,
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_numeric_imputed, y)

print(X_selected.head())

   gender  fever
0     1.0  103.0
1     1.0  100.0
2     1.0  101.0
3     0.0   98.0
4     0.0  101.0
